In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from dotenv import load_dotenv
import os
from pydantic import BaseModel
from typing import List, Optional

class Candidate(BaseModel):
    Candidate_name: str = None
    Years_of_experience: Optional[float] = None
    Current_role: Optional[str] = None
    Skills: Optional[List[str]] = None
    Highest_Education: Optional[str] = None

load_dotenv()  # Load environment variables from .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# Reuse the same Candidate Pydantic schema defined earlier

parser = PydanticOutputParser(pydantic_object=Candidate)

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="openai/gpt-oss-120b",
    temperature=0,
)

print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"Candidate_name": {"default": null, "title": "Candidate Name", "type": "string"}, "Years_of_experience": {"anyOf": [{"type": "number"}, {"type": "null"}], "default": null, "title": "Years Of Experience"}, "Current_role": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Current Role"}, "Skills": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Skills"}, "Highest_Education": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title

In [2]:

lc_prompt = PromptTemplate(
    template=(
        "You are a strict information extraction engine.\n"
        "Extract candidate information from the text below.\n"
        "If a field is not explicitly present in the text, you MUST return null for it "
        "(use an empty array [] for Skills if none are found). Do not guess or invent values.\n\n"
        "{format_instructions}\n\n"
        "Candidate text:\n{candidate_text}\n"
    ),
    input_variables=["candidate_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Chain: prompt -> llm -> pydantic output parser
lc_chain = lc_prompt | llm | parser


In [3]:
test_text = (
    "My name is Bavly, I am an AI engineer with 3 years of experience, "
    "graduated from Suez University. Through this experience I built a strong "
    "foundation in ML, CV applications, AI agents, and now I am a trainee with "
    "Digital Hub as an AI engineer."
)
lc_result_full = lc_chain.invoke({"candidate_text": test_text})

print("---- LANGCHAIN RESULT (full info) ----")
print(lc_result_full)
print(lc_result_full.model_dump_json(indent=2))

---- LANGCHAIN RESULT (full info) ----
Candidate_name='Bavly' Years_of_experience=3.0 Current_role='trainee' Skills=['ML', 'CV applications', 'AI agents'] Highest_Education='Suez University'
{
  "Candidate_name": "Bavly",
  "Years_of_experience": 3.0,
  "Current_role": "trainee",
  "Skills": [
    "ML",
    "CV applications",
    "AI agents"
  ],
  "Highest_Education": "Suez University"
}


In [4]:

no_experience_text = (
    "My name is Bavly, I am an AI engineer, graduated from Suez University. "
    "I built a strong foundation in ML, CV applications, and AI agents, "
    "and now I am a trainee with Digital Hub as an AI engineer."
)

lc_result_no_exp = lc_chain.invoke({"candidate_text": no_experience_text})

print("---- LANGCHAIN RESULT (years_of_experience omitted from input) ----")
print(lc_result_no_exp)
print(lc_result_no_exp.model_dump_json(indent=2))

assert lc_result_no_exp.Years_of_experience is None, "Expected Years_of_experience to be null!"
print("\nPASSED: Years_of_experience correctly returned as null when not mentioned in text.")


---- LANGCHAIN RESULT (years_of_experience omitted from input) ----
Candidate_name='Bavly' Years_of_experience=None Current_role='trainee with Digital Hub as an AI engineer' Skills=['ML', 'CV applications', 'AI agents'] Highest_Education='Suez University'
{
  "Candidate_name": "Bavly",
  "Years_of_experience": null,
  "Current_role": "trainee with Digital Hub as an AI engineer",
  "Skills": [
    "ML",
    "CV applications",
    "AI agents"
  ],
  "Highest_Education": "Suez University"
}

PASSED: Years_of_experience correctly returned as null when not mentioned in text.
